In [ ]:
from pydoc import text


y = []

carteiras_anuais = {}

melhor_pesos = None
historico = []
carteiras_criadas = []
for a in anos:
    y.append(a.split('-')[0])



print("=-"*48)
print("Anos totais: ",y)
print("=-"*48)

for an in anos:
    ano = an.split("-")[0]
    print("COMEÇANDO ANO NOVO: ",ano)
    #deletar modelo
    if 'model' in locals():
        del model
        print('Modelo Antigo deletado \n Iniciando Novo')
    else:
        print("Nao consta model")
        pass

    print("-"*15)
    print("Começando Modelo do ano: ", ano)
    print("="*6)

    try:
        score_usado = score_todos_anos[score_todos_anos.index.isin([ano])]
        # print(score_usado)
        print("Score Atualizado")
        

        ano_um = int(ano)+1
        df_usado = f'df_ativos_{ano_um}'
        retorno_usado = pd.DataFrame(eval(df_usado))
        retorno_usado = retorno_usado[lista_ativos_finais]
        print("Retornos atualizados")

        excesso_usado = dict_excesso[ano_um]
        print("Excessos Atualizados")
        
        sigma_usado = dict_sigma[ano_um]
        print("Sigmas Atualizados")
        
        print("-----")
    except Exception as e:
        print("ERRRRRRRRRRRROR")
        print(e)

    # if df_usado == 'df_ativos_2015':
    #     continue
    # else:
    print("# ------ CRIAÇÃO DO MODELO")
    print("## UTILIZANDO SCORE DO ANO DE: ",ano)
    print("## UTILIZANDO DADOS DE RETORNO DE: ",df_usado)
    print("## UTILIZANDO EXCESSO DO ANO DE: ",ano_um)
    print("## UTILIZANDO SIGMAS DO ANO DE: ",ano_um)

    model = pyo.ConcreteModel()

    #---------VARIÁVEIS-----------
    model.nome_ativos = pyo.Set(initialize = lista_ativos_finais)
    model.ativos = pyo.RangeSet(0, len(lista_ativos_finais)-1)
    model.dias = pyo.RangeSet(0, len(retorno_usado)-1)
    model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: retorno_usado.iloc[dia, ativo])    
    model.theta = pyo.Param(initialize=vb_theta)
    model.score = pyo.Param(model.ativos, initialize=lambda model,a: score_usado.iloc[0,a])
    model.cardinalidade_valor_max = pyo.Param(initialize=vb_cardinalidade_max)
    model.cardinalidade_valor_min = pyo.Param(initialize=vb_cardinalidade_min)
    model.peso_maximo = pyo.Param(initialize=vb_peso_maximo)
    model.peso_minimo = pyo.Param(initialize=vb_peso_minimo)
    model.x = pyo.Var(model.ativos, bounds=(0,1))
    model.y = pyo.Var(model.ativos, within=pyo.Binary)
    model.excesso = pyo.Param( model.ativos , initialize = lambda model,a: excesso_usado.mean().iloc[a])
    model.sigma = pyo.Param(model.ativos, model.ativos, initialize = lambda model,a,b: sigma_usado.iloc[a,b])
    model.s = pyo.Param(initialize = 2, mutable=True)
    model.r = pyo.Var(within=pyo.NonNegativeReals)
    
    #-------------------------------------- FUNÇÕES

    #=============================
    # Função Objetivo
    #=============================

    def func_objetivo_1(model):
        retorno_esperado = model.theta * sum(
            model.retornos_ativos[dia, a] * model.x[a] for a in model.ativos for dia in model.dias
        )
        # retorno_esperado = model.theta * model.r
        # A dúvida de qual retorno usar (Retorno BRUTO ou Excesso (retorno - selic))

        score_total = (1 - model.theta) * sum(
            model.x[a] * model.score[a] for a in model.ativos
        )
        return retorno_esperado + score_total
    model.obj1 = pyo.Objective(rule=func_objetivo_1, sense=pyo.maximize)

    #=============================
    # RESTRIÇÕES
    #=============================

    def def_r(model):
        return model.r == sum(model.excesso[a]*model.x[a]for a in model.ativos)
    model.const_def_r = pyo.Constraint(rule=def_r)

    ## modelos de Programação de Cone de Segunda Ordem (SOCP) 
    ## e Programação Quadrática com Restrições (QCP)
    def cone(model):
        return model.r**2 >= model.s**2 * sum(model.x[a]*model.sigma[a,b]*model.x[b] for a in model.ativos for b in model.ativos)
    model.constr_cone = pyo.Constraint(rule=cone)

    #REstricao 1 x só ativa se y = 1
    def restr_vinculo_x_y(model, a):
        return model.x[a] <= model.y[a]
    model.const_restr_vinculo_x_y = pyo.Constraint(model.ativos, rule=restr_vinculo_x_y)

    #peso maximo por acao
    def rule_peso_maximo(model, a):
        # return model.x[a] <= 1/model.cardinalidade_valor
        return model.x[a] <= model.peso_maximo
    model.const_peso_maximo = pyo.Constraint(model.ativos, rule=rule_peso_maximo)

    #peso minimo por acao
    def rule_peso_minimo(model, a):
        return model.x[a] >= model.peso_minimo * model.y[a]  # se y=1, então x >= 0.05
    model.const_peso_minimo = pyo.Constraint(model.ativos, rule=rule_peso_minimo)

    #Restrição 2 soma peso 1
    def soma_peso_1(model):
        return sum(model.x[a] for a in model.ativos) == 1
    model.const_soma_peso_1 = pyo.Constraint(rule=soma_peso_1)


    def cardinalidade_min(model):
        return sum(
            model.y[a] for a in model.ativos
            ) >= model.cardinalidade_valor_min
    model.const_cardinalidade_total_min = pyo.Constraint(rule=cardinalidade_min)

    def cardinalidade_max(model):
        return sum(
            model.y[a] for a in model.ativos
            ) <= model.cardinalidade_valor_max
    model.const_cardinalidade_total_max = pyo.Constraint(rule=cardinalidade_max)


    # NOTEBOOOK
    opt = SolverFactory('cplex', executable='C:\\CPLEX_Studio2211\\cplex\\bin\\x64_win64\\cplex.exe')

    # PC
    # opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')

    s_lo = 0.01        # <-- o Sharpe DIÁRIO 
    s_hi = 1   # teto: 2x o melhor ativo individual
    tol  = 0.001
    print("MODEL.S.VALUE => ",model.s.value)
    melhor_pesos = None
    historico = []
    carteiras_criadas = []
    # print(f"Valor a ser batido com S_LO: {s_lo} e S_HI: {s_hi} ... Diferença: {s_hi-s_lo}")
    print(f"Começando o WHILE do ano {ano}")
    while s_hi - s_lo > tol:
        s_a_ser_usado = 0.5 * (s_lo + s_hi)
        # print(s_a_ser_usado)
        model.s.value = s_a_ser_usado
        print("="*18)
        # print("Objetivo: (É o S lá da restrição de sharpe): ",s_a_ser_usado)
        res = opt.solve(model, load_solutions=False, tee=False)
        tc = res.solver.termination_condition
        print(f"Modelo Resolveu....Condição: ",tc)
        if tc == pyo.TerminationCondition.optimal:
            model.solutions.load_from(res)
            melhor_pesos = {a: pyo.value(model.x[a]) for a in model.ativos}
            # print({x:i for x,i in melhor_pesos.items() if i>0})
            s_lo = model.s.value
            print(f"Valor encontrado para o Sharpe: ",s_lo)
            # print(f"Valor FO: ",pyo.value(model.obj1))
            # print(f"Atualização de Valores para próxima rodagem: TC {tc} com S_LO: {s_lo} e S_HI: {s_hi} ... Diferença: {s_hi-s_lo}")
            # print("="*28)
            # for x,i in melhor_pesos.items():
            #     if i>0:

            #         dc = {
            #         'ativos_selecionados':melhor_pesos,
            #         'sharpe':pyo.value(model.s),
            #             }
            historico.append((s_a_ser_usado, 'viavel'))
            # carteiras_criadas.append(dc)
        elif tc in (pyo.TerminationCondition.infeasible,
                    pyo.TerminationCondition.infeasibleOrUnbounded):
            s_hi = s_a_ser_usado
            print(f"Atualização do S_HI : {s_hi}")
            print("="*28)

            historico.append((s_a_ser_usado, 'inviavel'))
        else:
            print(f'status inesperado em s={s_a_ser_usado:.4f}: {tc}')
            pass
    print("SAIU DO WHILE")
    print(f'Sharpe máximo (diário) ≈ {s_a_ser_usado:.4f}  |  anualizado ≈ {s_a_ser_usado*np.sqrt(252):.2f}')
    # print(historico)
    carteiras_anuais[int(ano)] = {
        'pesos':  melhor_pesos,
        'sharpe_anual': s_lo*np.sqrt(252),
    }
    print(f"{ano} -> {melhor_pesos}")
    print("+-="*30)
    if 'model' in locals():
        del model
        print('DELETADO DPS DO WHILE')
    else:
        print("Nao consta model")
        pass

    ## OQ ESTÀ FALTANDO?
    ## FAZER O SIGMA PARA TODOS


=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
Anos totais:  ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
COMEÇANDO ANO NOVO:  2015
Nao consta model
---------------
Começando Modelo do ano:  2015
Score Atualizado
Retornos atualizados
Excessos Atualizados
Sigmas Atualizados
-----
# ------ CRIAÇÃO DO MODELO
## UTILIZANDO SCORE DO ANO DE:  2015
## UTILIZANDO DADOS DE RETORNO DE:  df_ativos_2016
## UTILIZANDO EXCESSO DO ANO DE:  2016
## UTILIZANDO SIGMAS DO ANO DE:  2016
MODEL.S.VALUE =>  2
Começando o WHILE do ano 2015
Modelo Resolveu....Condição:  infeasible
Atualização do S_HI : 0.505
Modelo Resolveu....Condição:  optimal
Valor encontrado para o Sharpe:  0.2575
Modelo Resolveu....Condição:  infeasible
Atualização do S_HI : 0.38125
Modelo Resolveu....Condição:  infeasible
Atualização do S_HI : 0.3193

In [ ]:
ra = range(0,len(lista_ativos_finais))

for an in anos:
    an = int(an.split('-')[0])
    print(an)
    print([0 if carteiras_anuais[an]['pesos'][a] <0.02 else carteiras_anuais[an]['pesos'][a] for a in ra])

2015
[0, 0, 0.11494219991726529, 0, 0, 0, 0, 0, 0, 0, 0, 0.020000011218628922, 0, 0, 0, 0, 0.05474776433750346, 0, 0, 0.10004962235178518, 0, 0, 0, 0, 0, 0, 0, 0, 0.07944998466560435, 0, 0.14999999637487074, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.10577633196100296, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.14999983322796606, 0, 0, 0.07503472713875042, 0, 0, 0, 0, 0.14999938251446135, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2016
[0, 0, 0, 0.04507804023063836, 0, 0, 0, 0, 0, 0, 0, 0.03269722817118147, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.14999959922700845, 0, 0, 0, 0, 0, 0.09514279263400642, 0, 0, 0, 0.020000001432646613, 0, 0, 0, 0.14999999736901962, 0, 0.1110722529141741, 0, 0.1009630609824579, 0.14504667244645916, 0, 0, 0.14999999053108864, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2017
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.060127077165769094, 0, 0.12512020796917656, 0, 0, 0, 0, 0, 0, 0.14999903600572306, 0, 0.08415020951112488, 0, 0, 0, 0, 0, 0, 0, 0, 0.14640319676282815, 0.1499999849

In [ ]:
data = []
for carteira in carteiras_anuais:
    # print(carteiras_anuais[carteira])
    # print(len(carteiras_anuais[carteira]['pesos']))
    # print(len(lista_ativos_finais))
    # print(len(ra))
    final = pd.DataFrame({
        'peso_ativo': [carteiras_anuais[carteira]['pesos'][a] for a in ra],
        'ativado': [0 if carteiras_anuais[carteira]['pesos'][a] < vb_peso_minimo else 1 for a in ra],
    }, index=lista_ativos_finais)
    # print(len(final))
    df_final = final[final['ativado']!=0]
    data.append(df_final)

In [ ]:
np.round(data[0].sort_values(by='peso_ativo', ascending=False)['peso_ativo'],4)

FLRY3     0.1500
RADL3     0.1500
SUZB3     0.1500
AXIA3     0.1149
MGLU3     0.1058
CSMG3     0.1000
ENGI11    0.0794
RENT4     0.0750
CPFE3     0.0547
BRAP4     0.0200
Name: peso_ativo, dtype: float64